# lcnet

> PPLCNetV4 blocks for the medium detector and recognizer.


In [ ]:
#| default_exp layers.lcnet


#| export
The exported checkpoints are inference graphs. RepDWConv has already been fused into one depthwise convolution, and batch-norm on the backbone is folded into the convolution bias. The recognition neck keeps its own batch-norm, which is why `BatchNorm` lives next to the backbone that no longer uses it.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
def _unit_stride(stride: int | tuple[int, int]) -> bool:
    if isinstance(stride, int):
        return stride == 1
    return stride == (1, 1)


In [ ]:
from fastcore.test import test_eq

test_eq(_unit_stride(1), True)
test_eq(_unit_stride((1, 1)), True)
test_eq(_unit_stride((2, 1)), False)
_unit_stride((2, 1))


False

A residual is legal only when the stride is 1 in both directions and the channel count does not change. The recognizer's `(2, 1)` blocks are not residuals.


In [ ]:
#| export
#| notest
import numpy as np
from max.graph import TensorValue, Weight, ops
from max.nn.layer import Module
from max.nn.sequential import Sequential

from solid_lens.config import DEVICE, DTYPE, Block
from solid_lens.layers.activations import gelu, hardsigmoid, relu
from solid_lens.layers.conv import conv2d, nchw_to_nhwc, nhwc_to_nchw, same_padding


## Batch-norm


In [ ]:
#| export
#| notest
class BatchNorm(Module):
    def __init__(self, channels: int, eps: float) -> None:
        super().__init__()
        self.eps = eps
        self.weight = Weight("weight", DTYPE, (channels,), DEVICE)
        self.bias = Weight("bias", DTYPE, (channels,), DEVICE)
        self.running_mean = Weight("running_mean", DTYPE, (channels,), DEVICE)
        self.running_var = Weight("running_var", DTYPE, (channels,), DEVICE)

    def __call__(self, x: TensorValue) -> TensorValue:
        shape = [1, int(self.weight.shape[0])] + [1] * (x.rank - 2)
        scale = ops.reshape(self.weight, shape)
        bias = ops.reshape(self.bias, shape)
        mean = ops.reshape(self.running_mean, shape)
        var = ops.reshape(self.running_var, shape)
        inv_std = ops.rsqrt(var + ops.constant(np.float32(self.eps), DTYPE, device=DEVICE))
        return (x - mean) * inv_std * scale + bias


This is inference batch-norm: the running statistics are weights, not buffers that update. LightSVTR uses it. The backbone does not, because those batch-norms were folded into the convolution bias before export.


## Squeeze-excite


In [ ]:
#| export
#| notest
class SqueezeExcite(Module):
    def __init__(
        self,
        channels: int,
        alpha: float,
        *,
        reduction: int = 4,
        residual: bool = False,
    ) -> None:
        super().__init__()
        hidden = channels // reduction
        self.alpha = alpha
        self.residual = residual
        self.down = conv2d(channels, hidden, 1, bias=True)
        self.up = conv2d(hidden, channels, 1, bias=True)

    def __call__(self, x: TensorValue) -> TensorValue:
        pooled = ops.mean(ops.mean(x, axis=2), axis=3)
        gate = hardsigmoid(self.up(relu(self.down(pooled))), self.alpha)
        scaled = x * gate
        if self.residual:
            return x + scaled
        return scaled


## The block


In [ ]:
#| export
#| notest
class LCNetV4Block(Module):
    "Token mixer (depthwise conv, optional SE) plus a 2x channel mixer."

    def __init__(self, spec: Block, se_alpha: float, *, bias: bool) -> None:
        super().__init__()
        kernel, in_channels, out_channels, stride, use_se = spec
        self.residual = _unit_stride(stride) and in_channels == out_channels
        self.dw = conv2d(
            in_channels,
            in_channels,
            kernel,
            stride=stride,
            padding=same_padding(kernel),
            groups=in_channels,
            bias=bias,
        )
        self.se = SqueezeExcite(in_channels, se_alpha) if use_se else None
        hidden = in_channels * 2
        self.expand = conv2d(in_channels, hidden, 1, bias=bias)
        self.compress = conv2d(hidden, out_channels, 1, bias=bias)

    def __call__(self, x: TensorValue) -> TensorValue:
        mixed = self.dw(x)
        if self.se is not None:
            mixed = self.se(mixed)
        channels = gelu(self.expand(mixed))
        channels = self.compress(channels)
        if self.residual:
            return mixed + channels
        return channels


The residual is added to the depthwise output, not to the block input. The channel mixer is an expansion by 2, a GELU, and a projection. Submodule order here is the checkpoint order: depthwise, optional SE, expand, compress.


In [ ]:
#| export
#| notest
def stage(blocks: list[Block], se_alpha: float, *, bias: bool) -> Sequential:
    return Sequential(
        [LCNetV4Block(spec, se_alpha, bias=bias) for spec in blocks]
    )


## Stem


In [ ]:
#| export
#| notest
class DetStem(Module):
    "Multi-branch stem. Total stride is 4."

    def __init__(self, mid: int, out: int) -> None:
        super().__init__()
        self.stem1 = conv2d(3, mid, 3, stride=2, padding=1, bias=True)
        self.stem2a = conv2d(mid, mid // 2, 2, stride=1, padding=(0, 1, 0, 1), bias=True)
        self.stem2b = conv2d(mid // 2, mid, 2, stride=1, padding=(0, 1, 0, 1), bias=True)
        self.stem3 = conv2d(mid * 2, mid, 3, stride=2, padding=1, bias=True)
        self.stem4 = conv2d(mid, out, 1, bias=True)

    def __call__(self, x: TensorValue) -> TensorValue:
        x = relu(self.stem1(x))
        branch = relu(self.stem2b(relu(self.stem2a(x))))
        pooled = _max_pool_same_upper(x)
        merged = ops.concat([pooled, branch], axis=1)
        return relu(self.stem4(relu(self.stem3(merged))))


The stem's total stride is 4: one stride-2 convolution, then a stride-2 convolution after the branches merge. The 2×2 branch is padded `(0, 1, 0, 1)`, which is the asymmetric SAME the exporter used. The other branch is a 2×2 max pool with the same asymmetry. Both recognizer and detector use this stem.


In [ ]:
#| export
#| notest
def _max_pool_same_upper(x: TensorValue) -> TensorValue:
    "Kernel-2, stride-1 SAME_UPPER max pool. Padding is asymmetric."
    x = ops.pad(x, [0, 0, 0, 0, 0, 1, 0, 1])
    y = ops.max_pool2d(nchw_to_nhwc(x), kernel_size=(2, 2), stride=1, padding=0)
    return nhwc_to_nchw(y)
